# Imported from `mc_startup.ipynb`

Cells below are copied in the requested order. Redundancies are preserved.

In [ ]:
import mat73
from matplotlib import gridspec


import numpy as np  #The basic python scientific package
import netCDF4 as nc4
import matplotlib.pyplot as plt #Simple visualisation package
import seaborn as sns #Datascience visualisation package
import pandas as pd #Datascience package for handling data
from datetime import datetime
from time import time

from windrose import WindroseAxes
import math
import gsw
import seawater as sw
from pathlib import Path

from scipy import stats
from scipy.optimize import curve_fit

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

from IPython.display import display, Math, Latex

from scipy import signal

from scipy.stats import t

import random
from random import randrange
from pandas import Series
from statsmodels.tsa.seasonal import seasonal_decompose

LAT_MC = 40.80
LON_MC = 14.25

LAT_BC = 40.5
LON_BC = 14.0

LAT_TY = 39.25
LON_TY = 13.5

In [ ]:
from pandas import read_csv
from pandas.plotting import scatter_matrix
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Imported from `myfunctions.ipynb`

Cells below are copied in the requested order. Redundancies are preserved.

# FUNCTIONS

In [ ]:
def find_nearest3(array, values):
    values = np.atleast_1d(values)
    indices = np.abs(np.int64(np.subtract.outer(array, values))).argmin(0)
    out = array[indices]
    return indices

In [ ]:
def center_array_indices(x_in, N):
    l = len(x_in)
    M = np.round((N-1)/2)+1
    x_out = np.array([np.nan]*l)
    x_out[int(N-M):int(l-M)] = x_in[N:l]
    return x_out

In [ ]:
def interpholes(x_in, y_in, x_out):
    ff = np.where((np.isfinite(y_in)))[0]
    y_out = np.interp(x_out, x_in[ff], y_in[ff])   
    return y_out

In [ ]:
def beaning_weekly(y_in, marker_range, marker1, marker2): 
    l = len(marker_range)
    x_out = np.empty(l*52) * np.NaN
    x_out2 = np.empty(l*52) * np.NaN
    x_out3 = np.empty(l*52) * np.NaN
    x_out4 = np.empty(l*52) * np.NaN
    y_out = np.empty(l*52) * np.NaN 
    k = -1
    for i in range(0,l): 
        for w in range(0,53):
            k = k+1
            v = marker_range[i]
            x_out2[k] = v 
            x_out3[k] = w
            x_out4[k] = k
            x_out[k] = v + w/53
            fiw = np.where((marker1 == v) & (marker2 == w))[0]
            if len(fiw)>0:
                y_out[k] = np.nanmean(y_in[fiw])
    return x_out, x_out2, x_out3, x_out4, y_out

In [ ]:
def moving_average(x, w):
    return np.convolve(x, np.ones(w), 'valid') / w

In [ ]:
def beaning_plus(x_in, marker_range, marker):
    l = len(marker_range)
    x_out = np.empty(l) * np.NaN
    xx_out = np.empty(l) * np.NaN
    std_out = np.empty(l) * np.NaN
    stde_out = np.empty(l) * np.NaN
    N_out = np.empty(l) * np.NaN
    
    for i in range(0,l): 
        v = marker_range[i]
        f = np.where((marker == v))[0]
        if len(f)>0:
            N_out[i] = len(f)
            std_out[i] = np.nanstd(x_in[f])
            stde_out[i] = np.nanstd(x_in[f])/np.sqrt(len(f))
            x_out[i] = np.nanmean(x_in[f])
            xx_out[i] = np.nanmedian(x_in[f])
    return x_out, std_out, stde_out, N_out, xx_out

In [ ]:
def beaning(x_in, marker_range, marker):
    l = len(marker_range)
    x_out = np.empty(l) * np.NaN
    for i in range(0,l): 
        v = marker_range[i]
        f = np.where((marker == v))[0]
        if len(f)>0:
            x_out[i] = np.nanmean(x_in[f])
    return x_out

def integrate(X, z1, z2):
    sxx = X.shape
    sx1 = sxx[0]
    sx2 = sxx[1]
    x_out  = np.empty(sx2) * np.NaN
    for i in range(0,sx2): 
        z = np.arange(int(z1[i]),int(z2[i])+1)
        v = X[z,i]   
        fv = np.where((np.isfinite(v)))[0]
        if len(np.isfinite(v) == True) > 0:
            x_out[i] = np.nansum(X[fv,i]) 
    x_out[x_out==0] = np.nan 
    return x_out

In [ ]:
def datenum(d):
    return 366 + d.toordinal() + (d - dt.fromordinal(d.toordinal())).total_seconds()/(24*60*60)
#d = dt.strptime('2019-2-1 12:24','%Y-%m-%d %H:%M')
#datenum(d)


In [ ]:
from scipy.stats import t

def make_linregress(y_in, marker_1, marker_2, range_marker_1, range_marker_2):
    l = len(range_marker_1)
    slpe = np.array([np.NaN] * l)
    rsqu = np.array([np.NaN] * l)
    pval = np.array([np.NaN] * l)
    stde = np.array([np.NaN] * l)
    intr = np.array([np.NaN] * l)
    slpe_ci = np.array([np.NaN] * l)
    for m in range(range_marker_1[0],range_marker_1[l-1]+1):
        fm = np.where((marker_1 == m))[0]
        y = beaning(y_in[fm], range_marker_2, marker_2[fm]) #  - - - y
        x = range_marker_2
        ff = np.where(np.isfinite(y))[0]
        if len(ff)>0:
            X = np.reshape(x, (len(x), 1))
            model = LinearRegression()
            model.fit(X, y)
            trend = model.predict(X) #  - - - - - - - - - - - - - - - - trend
            detrended = [y[i]-trend[i] for i in range(0, len(y))]
            res = stats.linregress(x,y) #  - - - - - - - - - - - - - - - stats
            trend = res.slope * x + res.intercept
            slpe[m-1] = res.slope
            rsqu[m-1] = res.rvalue**2
            pval[m-1] = res.pvalue
            stde[m-1] = res.stderr
            intr[m-1] = res.intercept
            #
            tinv = lambda p, df: abs(t.ppf(p/2, df))
            ts = tinv(0.05, len(x)-2)
            slpe_ci[m-1] = ts*res.stderr
        
    return slpe, rsqu, pval, stde, intr, slpe_ci

In [ ]:
def make_split_linregress(y_in, marker_1, val_markers):
    l = len(val_markers)-1
    slpe = np.array([np.NaN] * l)
    rsqu = np.array([np.NaN] * l)
    pval = np.array([np.NaN] * l)
    stde = np.array([np.NaN] * l)
    intr = np.array([np.NaN] * l)
    slpe_ci = np.array([np.NaN] * l)
    
    for seg in range(0,l):
        # print(seg)
        f = np.where((marker_1 >= val_markers[seg]) & (marker_1 < val_markers[seg+1]))[0]
        x = marker_1[f]
        y = y_in[f]
        ff = np.where(np.isfinite(y))[0]
        if len(ff)>0:
            x = x[ff]
            y = y[ff]
            X = np.reshape(x, (len(x), 1))
            model = LinearRegression()
            model.fit(X, y)
            trend = model.predict(X) #  - - - - - - - - - - - - - - - - trend
            detrended = [y[i]-trend[i] for i in range(0, len(y))]
            res = stats.linregress(x,y) #  - - - - - - - - - - - - - - - stats
            trend = res.slope * x + res.intercept
            slpe[seg] = res.slope
            rsqu[seg] = res.rvalue**2
            pval[seg] = res.pvalue
            stde[seg] = res.stderr
            intr[seg] = res.intercept
            #
            tinv = lambda p, df: abs(t.ppf(p/2, df))
            ts = tinv(0.05, len(x)-2)
            slpe_ci[seg] = ts*res.stderr       
    return slpe, rsqu, pval, stde, intr, slpe_ci, y_in

In [ ]:

# -*- coding: utf-8 -*-
#
# windstress.py
#
# purpose:
# author:   Filipe P. A. Fernandes
# e-mail:   ocefpaf@gmail
# web:      http://ocefpaf.github.io/
# created:  21-Aug-2013
# modified: Mon 21 Jul 2014 12:16:28 PM BRT
#
# obs:
#

import numpy as np

# from .constants import kappa, Charnock_alpha, g, R_roughness

# ---- meteorological constants
kappa = 0.4  # NOTE: 0.41
""" von Karman's constant """

charn = Charnock_alpha = 0.011  # NOTE: 0.018
""" Charnock constant. For determining roughness length at sea given friction
velocity, used in Smith formulas for drag coefficient and also in Fairall and
Edson. Ese alpha = 0.011 for open-ocean and alpha = 0.018 for fetch-limited
(coastal) regions."""

R_roughness = 0.11
""" limiting roughness Reynolds for aerodynamically smooth flow """



#from .atmosphere import visc_air

def visc_air(Ta):
    """Computes the kinematic viscosity of dry air as a function of air
    temperature
    Parameters
    ----------
    Ta : array_like
         air temperature [:math:`^\\circ` C]
    Returns
    -------
    visa : array_like
           [m :sup:`2` s :sup:`-1`]
    See Also
    --------
    hfbulktc, cdn
    sw.visc
    Notes
    -----
    sw.visc from python seawater package
    Examples
    --------
    >>> from airsea import atmosphere as asea
    >>> asea.visc_air([[0.1, 5., 15],[22.8, 28.9, 31.4]])
    array([[  1.32686758e-05,   1.36964784e-05,   1.45857532e-05],
           [  1.52942886e-05,   1.58573695e-05,   1.60903922e-05]])
    References
    ----------
    .. [1] Andreas (1989), CRREL Report 89-11.
    Modifications: Original from COARE 3.0
    11/26/2010: Filipe Fernandes, Python translation.
    """
    # convert input to numpy array
    Ta = np.asarray(Ta)

    visa = 1.326e-5 * (1 + 6.542e-3 * Ta + 8.301e-6 * Ta ** 2 - 4.84e-9 *
                       Ta ** 3)
    return visa



def cdn(sp, z, drag='largepond', Ta=10):
    """Computes neutral drag coefficient.
    Methods available are: Large & Pond (1981),  Vera (1983) or Smith (1988)
    Parameters
    ----------
    sp : array_like
         wind speed [m s :sup:`-1`]
    z : float, array_like
        measurement height [m]
    drag : str
           neutral drag by:
           'largepond' <-- default
           'smith'
           'vera'
    Ta : array_like, optional for drag='smith'
         air temperature [:math:`^\\circ` C]
    Returns
    -------
    cd : float, array_like
         neutral drag coefficient at 10 m
    u10 : array_like
          wind speed at 10 m [m s :sup:`-1`]
    See Also
    --------
    stress, spshft, visc_air
    Notes
    -----
    Vera (1983): range of fit to data is 1 to 25 [m s :sup:`-1`].
    Examples
    --------
    >>> from airsea import windstress as ws
    >>> ws.cdn([10., 0.2, 12., 20., 30., 50.], 10)
    (array([ 0.00115,  0.00115,  0.00127,  0.00179,  0.00244,  0.00374]),
     array([ 10. ,   0.2,  12. ,  20. ,  30. ,  50. ]))
    >>> ws.cdn([10., 0.2, 12., 20., 30., 50.], 15, 'vera')
    (array([ 0.00116157,  0.01545237,  0.00126151,  0.00174946,  0.00242021,
            0.00379521]),
     array([  9.66606155,   0.17761896,  11.58297824, 19.18652915,
            28.5750255 ,  47.06117334]))
    >>> ws.cdn([10., 0.2, 12., 20., 30., 50.], 20, 'smith', 20.)
    (array([ 0.00126578,  0.00140818,  0.00136533,  0.00173801,  0.00217435,
            0.00304636]),
     array([  9.41928554,   0.18778865,  11.27787697,  18.65250005,
            27.75712916,  45.6352786 ]))
    References
    ----------
    .. [1] Large and Pond (1981), J. Phys. Oceanog., 11, 324-336.
    .. [2] Smith (1988), J. Geophys. Res., 93, 311-326.
    .. [3] E. Vera (1983) FIXME eqn. 8 in Large, Morzel, and Crawford (1995),
    J. Phys. Oceanog., 25, 2959-2971.
    Modifications: Original from AIR_SEA TOOLBOX, Version 2.0
    03-08-1997: version 1.0
    08-26-1998: version 1.1 (vectorized by RP)
    08-05-1999: version 2.0
    11-26-2010: Filipe Fernandes, Python translation.
    """
    # convert input to numpy array
    sp, z, Ta = np.asarray(sp), np.asarray(z), np.asarray(Ta)

    tol = 0.00001  # Iteration end point.

    if drag == 'largepond':
        a = np.log(z / 10.) / kappa  # Log-layer correction factor.
        u10o = np.zeros(sp.shape)
        cd = 1.15e-3 * np.ones(sp.shape)
        u10 = sp / (1 + a * np.sqrt(cd))
        ii = np.abs(u10 - u10o) > tol

        while np.any(ii):
            u10o = u10
            cd = (4.9e-4 + 6.5e-5 * u10o)  # Compute cd(u10).
            cd[u10o < 10.15385] = 1.15e-3
            u10 = sp / (1 + a * np.sqrt(cd))  # Next iteration.
            # Keep going until iteration converges.
            ii = np.abs(u10 - u10o) > tol

    elif drag == 'smith':
        visc = visc_air(Ta)

        # Remove any sp==0 to prevent division by zero
        # i = np.nonzero(sp == 0)
        # sp[i] = 0.1 * np.ones(len(i)) FIXME

        # initial guess
        ustaro = np.zeros(sp.shape)
        ustarn = 0.036 * sp

        # iterate to find z0 and ustar
        ii = np.abs(ustarn - ustaro) > tol
        while np.any(ii):
            ustaro = ustarn
            z0 = Charnock_alpha * ustaro ** 2 / g + R_roughness * visc / ustaro
            ustarn = sp * (kappa / np.log(z / z0))
            ii = np.abs(ustarn - ustaro) > tol

        sqrcd = kappa / np.log(10. / z0)
        cd = sqrcd ** 2
        u10 = ustarn / sqrcd
    elif drag == 'vera':
        # constants in fit for drag coefficient
        A = 2.717e-3
        B = 0.142e-3
        C = 0.0764e-3

        a = np.log(z / 10.) / kappa  # Log-layer correction factor.
        # Don't start iteration at 0 to prevent blowups.
        u10o = np.zeros(sp.shape) + 0.1
        cd = A / u10o + B + C * u10o
        u10 = sp / (1 + a * np.sqrt(cd))

        ii = np.abs(u10 - u10o) > tol
        while np.any(ii):
            u10o = u10
            cd = A / u10o + B + C * u10o
            u10 = sp / (1 + a * np.sqrt(cd))  # Next iteration.
            # Keep going until iteration converges.
            ii = np.abs(u10 - u10o) > tol
    else:
        print('Unknown method')  # FIXME: raise a proper python error.

    return cd, u10


def spshft(sp, z1, z2, drag='largepond', Ta=10.):
    """Adjusts wind speed from height z1 to z2. Methods available are: Large &
    Pond (1981),  Vera (1983) or Smith (1988).
    Parameters
    ----------
    sp : array_like
          wind speed [m s :sup:`-1`]
    z1 : float
         measurement height [m]
    z2 : float
         desired height [m]
    drag : str
           neutral drag by:
           'largepond' <-- default
           'smith'
           'vera'
    Ta : array_like
         air temperature [:math:`^\\circ` C]
    Returns
    -------
    sp_adj : array_like
          predicted wind speed [m s :sup:`-1`]
    ustar : array_like
            friction velocity [m s :sup:`-1`]
    See Also
    --------
    cdn
    Examples
    --------
    >>> from airsea import windstress as ws
    >>> ws.spshft([10., 0.2, 12., 20., 30., 50.], 10, 10)[0]
    array([ 10. ,   0.2,  12. ,  20. ,  30. ,  50. ])
    >>> from airsea import windstress as ws
    >>> ws.spshft([10., 0.2, 12., 20., 30., 50.], 10, 8, 'smith', 20)
    (array([  9.79908171,   0.19583568,  11.74922628,  19.52618419,
            29.20068179,  48.40456082]), array([ 0.3601597 ,  0.00746483,  0.44952896,  0.84934708,  1.43283229,
            2.8599333 ]))
    >>> ws.spshft([10., 0.2, 12., 20., 30., 50.], 15, 10, 'vera')
    (array([  9.66606155,   0.17761896,  11.58297824,  19.18652915,
            28.5750255 ,  47.06117334]), array([ 0.32943742,  0.02207938,  0.41140089,  0.80250639,  1.40576781,
            2.89921535]))
    References
    ----------
    .. [1] Large and Pond (1981), J. Phys. Oceanog., 11, 324-336.
    .. [2] Smith (1988), J. Geophys. Res., 93, 311-326.
    .. [3] E. Vera (1983) FIXME eqn. 8 in Large, Morzel, and Crawford (1995),
    J. Phys. Oceanog., 25, 2959-2971.
    Modifications: Original from AIR_SEA TOOLBOX, Version 2.0
    03-08-1997: version 1.0
    08-27-1998: version 1.1 (revised to use cdn* efficiently by RP)
    08-05-1999: version 2.0
    11-26-2010: Filipe Fernandes, Python translation.
    """

    z1, z2 = np.asarray(z1), np.asarray(z2)
    sp, Ta = np.asarray(sp), np.asarray(Ta)

    # Find cd and ustar.
    if drag == 'largepond':
        cd10, sp10 = cdn(sp, z1, 'largepond')
    elif drag == 'smith':
        cd10, sp10 = cdn(sp, z1, 'smith', Ta)
    elif drag == 'vera':
        cd10, sp10 = cdn(sp, z1, 'vera')
    else:
        print('Unknown method')  # FIXME: raise a proper python error!

    ustar = np.sqrt(cd10) * sp10
    sp_adj = sp10 + ustar * np.log(z2 / 10.) / kappa
    return sp_adj, ustar


def stress(sp, z=10., drag='largepond', rho_air=1.22, Ta=10.):
    """Computes the neutral wind stress.
    Parameters
    ----------
    sp : array_like
         wind speed [m s :sup:`-1`]
    z : float, array_like, optional
        measurement height [m]
    rho_air : array_like, optional
           air density [kg m :sup:`-3`]
    drag : str
           neutral drag by:
           'largepond' <-- default
           'smith'
           'vera'
    Ta : array_like, optional
         air temperature [:math:`^\\circ` C]
    Returns
    -------
    tau : array_like
          wind stress  [N m :sup:`-2`]
    See Also
    --------
    cdn
    Examples
    --------
    >>> from airsea import windstress as ws
    >>> ws.stress([10., 0.2, 12., 20., 30., 50.], 10)
    array([  1.40300000e-01,   5.61200000e-05,   2.23113600e-01,
             8.73520000e-01,   2.67912000e+00,   1.14070000e+01])
    >>> kw = dict(rho_air=1.02, Ta=23.)
    >>> ws.stress([10., 0.2, 12., 20., 30., 50.], 15, 'smith', **kw)
    array([  1.21440074e-01,   5.32531576e-05,   1.88322389e-01,
             6.62091968e-01,   1.85325310e+00,   7.15282267e+00])
    >>> ws.stress([10., 0.2, 12., 20., 30., 50.], 8, 'vera')
    array([  1.50603698e-01,   7.16568379e-04,   2.37758830e-01,
             9.42518454e-01,   3.01119044e+00,   1.36422742e+01])
    References
    ----------
    .. [1] Large and Pond (1981), J. Phys. Oceanog., 11, 324-336.
    .. [2] Smith (1988), J. Geophys. Res., 93, 311-326.
    .. [3] E. Vera (1983) FIXME eqn. 8 in Large, Morzel, and Crawford (1995),
    J. Phys. Oceanog., 25, 2959-2971.
    Modifications: Original from AIR_SEA TOOLBOX, Version 2.0
    03-08-1997: version 1.0
    08-26-1998: version 1.1 (revised by RP)
    04-02-1999: versin 1.2 (air density option added by AA)
    08-05-1999: version 2.0
    11-26-2010: Filipe Fernandes, Python translation.
    """
    z, sp = np.asarray(z), np.asarray(sp)
    Ta, rho_air = np.asarray(Ta), np.asarray(rho_air)

    # Find cd and ustar.
    if drag == 'largepond':
        cd, sp = cdn(sp, z, 'largepond')
    elif drag == 'smith':
        cd, sp = cdn(sp, z, 'smith', Ta)
    elif drag == 'vera':
        cd, sp = cdn(sp, z, 'vera')
    else:
        print('Unknown method')  # FIXME: raise a proper python error

    tau = rho_air * (cd * sp ** 2)

    return tau


In [ ]:
def average_on_unique(x_in, marker):
    ux, iu = np.unique(marker, return_index=True)
    l = len(iu)
    x_out = np.empty(l) * np.NaN
    for i in range(0,l): 
        f = np.where((marker == ux[i]))[0]
        if len(f)>0:
            x_out[i] = np.nanmean(x_in[f])
    return x_out, ux, iu

# Imported from `mycolors.ipynb`

Cells below are copied in the requested order. Redundancies are preserved.

COLORS

In [ ]:
clr_spring = '#30b874'
clr_summer = '#f05668'
clr_autumn = '#f29563'
clr_winter = '#3e89de'

clr_spring_I = '#87edba'
clr_summer_I = '#ffadb7'
clr_autumn_I = '#ffbe9c'
clr_winter_I = '#91c5ff'

clr_spring_II = '#36a36d'
clr_summer_II = '#f55f72'
clr_autumn_II = '#f58347'
clr_winter_II = '#2e77c9'


clr_ia = '#646a70'

clr_blackish = '#333333'
clr_therma = '#e64c4c'
clr_saline = '#2148bf'
clr_fresh = '#39b0db'

clr_bckgnd = '#c4c4c4'

clr_bckgnd1 = '#f2d563'
clr_bckgnd2 = '#a9ebc8'

clr_period1 = '#d92967'
clr_period2 = '#61ad85'

clr_decadeI = '#d92967'
clr_decadeII = '#61ad85'

clr_decadeI = '#91b3d9'
clr_decadeII = '#2e77c9'







clr_stable = 'k'
clr_unstab = 'k'

clr_heat= '#ed53a8'
clr_sst = '#ed53a8'
clr_wind = '#ed53a8'
clr_lang = '#30cfa9'
clr_conv = '#f7de39'

clr_tempopasse = '#333333'

# Imported from `my_functions_RF_CHL_GON.ipynb`

Cells below are copied in the requested order. Redundancies are preserved.

In [ ]:
clr_rcp45 = '#fcac6f'
clr_rcp85 = '#fa6d4d'

clr_chl = '#0cb053'
clr_sal = '#416ee0'
clr_mld = '#4a4a4a'
clr_cri = '#2bb3ed'


In [ ]:
import numpy as np

def stats_zrange_series(DF, z, z1, z2):
    """
    DF : array (Nz, Nt)
    z  : array (Nz,)
    z1, z2 : scalaires ou vecteurs (Nt,)
    """

    DF = np.asarray(DF, dtype=float)
    z = np.asarray(z, dtype=float)
    Nt = DF.shape[1]

    # z croissant
    if z[0] > z[-1]:
        z = z[::-1]
        DF = DF[::-1, :]

    # scalaires -> vecteurs constants
    z1 = np.full(Nt, z1, dtype=float) if np.isscalar(z1) else np.asarray(z1, dtype=float)
    z2 = np.full(Nt, z2, dtype=float) if np.isscalar(z2) else np.asarray(z2, dtype=float)

    mean = np.full(Nt, np.nan)
    cum = np.full(Nt, np.nan)
    cum_interp = np.full(Nt, np.nan)

    for t in range(Nt):

        za = min(z1[t], z2[t])
        zb = max(z1[t], z2[t])

        mask = (z >= za) & (z <= zb)

        if np.sum(mask) < 1:
            continue

        zz = z[mask]
        xx = DF[mask, t]

        # moyenne en ignorant NaN
        mean[t] = np.nanmean(xx)

        # cumul brut en ignorant NaN
        if len(zz) >= 2:
            dz = np.gradient(zz)
            cum[t] = np.nansum(xx * dz)
        else:
            cum[t] = np.nan

        # cumul avec interpolation verticale
        good = ~np.isnan(xx)

        if np.sum(good) >= 2:
            xx_i = np.interp(zz, zz[good], xx[good])
            cum_interp[t] = np.sum(xx_i * dz)
        else:
            cum_interp[t] = np.nan

    return mean, cum, cum_interp

In [ ]:
#import inspect
#lines = inspect.getsource(beaning)
#print(lines)

In [ ]:


from scipy.signal import butter, filtfilt

try:
    from scipy.signal.windows import tukey
except:
    from scipy.signal import tukey
    
def filtre_butter(x, cutoff, fs, ordre=3):
    b, a = butter(ordre, cutoff / (0.5 * fs), btype='low')
    return filtfilt(b, a, x)


def butter_lowpass(cutoff, nyq_freq, order=2):
    normal_cutoff = float(cutoff) / nyq_freq
    b, a = signal.butter(order, normal_cutoff, btype='lowpass')
    return b, a

def butter_lowpass_filter(data, cutoff_freq, nyq_freq, order=2):
    # Source: https://github.com/guillaume-chevalier/filtering-stft-and-laplace-transform
    b, a = butter_lowpass(cutoff_freq, nyq_freq, order=order)
    y = signal.filtfilt(b, a, data)
    return y


def butter_hipass(cutoff, nyq_freq, order=2):
    normal_cutoff = float(cutoff) / nyq_freq
    b, a = signal.butter(order, normal_cutoff, btype='highpass')
    return b, a

def butter_hipass_filter(data, cutoff_freq, nyq_freq, order=2):
    # Source: https://github.com/guillaume-chevalier/filtering-stft-and-laplace-transform
    b, a = butter_hipass(cutoff_freq, nyq_freq, order=order)
    y = signal.filtfilt(b, a, data)
    return y

def dataframe_remove_nan(df):
    for c in range(0,df.shape[1]):
        
        x = np.array(df.iloc[:,c])
        if isinstance(x[0],float)==True:
            f = np.where(np.isfinite(x)==True)[0]
            if len(f)>0:
            #    print(len(f))
                df = df.iloc[f,:]
                df = df.set_index(np.arange(0,df.shape[0]))          
    return df

def dataframe_merger(df1,marker1,df2,marker2): 
    df12 = pd.DataFrame()
    
    list1 = list(df1.columns)
    list2 = list(df2.columns)
    list(set(list2).intersection(list1))

    common_marker = np.intersect1d(marker1, marker2)
    F2 = []
    for i in range(0,len(common_marker)):
        f2 = np.where(marker2 == common_marker[i])[0]
        if len(f2)>0:
            F2.append(f2[0])
    F2 = np.array(F2)
    
    F1 = []
    for i in range(0,len(common_marker)):
        f1 = np.where(marker1 == common_marker[i])[0]
        if len(f1)>0:
            F1.append(f1[0])
    F1 = np.array(F1)
    
    for c in range(0,df1.shape[1]):
        colname = list1[c]
        df12[colname] = np.array(df1.iloc[F1,c])
    
    
    for c in range(0,df2.shape[1]):
        colname = list2[c]
        if (colname in list1)==True:
            colname = colname+'*'
        df12[colname] = np.array(df2.iloc[F2,c])
    
    return df12

In [ ]:
def normalize_01(x):
    x01 = (x - np.nanmin(x)) / np.abs( np.nanmax(x) - np.nanmin(x))
    return x01

def normalize_11(x):
    x11 = (x) / np.abs( np.nanmax(x) - np.nanmin(x))
    return x11


from scipy.stats import t

def simple_linregress(x,y):
    f = np.where(np.isfinite(x*y)==1)[0]
    if len(f)>0:
        x = x[f]
        y = y[f]
        X = np.reshape(x, (len(x), 1))
        model = LinearRegression()
        model.fit(X, y)
        trend = model.predict(X) #  - - - - - - - - - - - - - - - - trend
        detrended = [y[i]-trend[i] for i in range(0, len(y))]
        res = stats.linregress(x,y) #  - - - - - - - - - - - - - - - stats
        #trend = res.slope * x + res.intercept
        #slpe = res.slope
        #rsqu = res.rvalue**2
        pval = res.pvalue
        #stde = res.stderr
        #intr = res.intercept
        ccoef =  np.corrcoef((x,y))
        #ccoef_detrended =  np.corrcoef((x,detrended))
        #
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, len(x)-2)
        slpe_ci = ts*res.stderr

        return pval, ccoef[1,0], slpe_ci, ts
    
def simple_linregress_full(x,y):
    f = np.where(np.isfinite(x*y)==1)[0]
    if len(f)>0:
        x = x[f]
        y = y[f]
        X = np.reshape(x, (len(x), 1))
        model = LinearRegression()
        model.fit(X, y)
        trend = model.predict(X) #  - - - - - - - - - - - - - - - - trend
        detrended = [y[i]-trend[i] for i in range(0, len(y))]
        res = stats.linregress(x,y) #  - - - - - - - - - - - - - - - stats
        #trend = res.slope * x + res.intercept
        slpe = res.slope
        rsqu = res.rvalue**2
        pval = res.pvalue
        stde = res.stderr
        intr = res.intercept
        ccoef =  np.corrcoef((x,y))
        #ccoef_detrended =  np.corrcoef((x,detrended))
        #
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, len(x)-2)
        slpe_ci = ts*res.stderr

        return pval, ccoef[1,0], slpe,  intr, rsqu, slpe_ci, ts
    
    


def beaning_median(x_in, marker_range, marker):
    l = len(marker_range)
    x_out = np.empty(l) * np.NaN
    for i in range(0,l): 
        v = marker_range[i]
        f = np.where((marker == v))[0]
        if len(f)>0:
            x_out[i] = np.nanmedian(x_in[f])
    return x_out

def beaning_mean(x_in, marker_range, marker):
    l = len(marker_range)
    x_out = np.empty(l) * np.NaN
    for i in range(0,l): 
        v = marker_range[i]
        f = np.where((marker == v))[0]
        if len(f)>0:
            x_out[i] = np.nanmean(x_in[f])
    return x_out


def get_metrucs(A,P,ROUNDORNOT,NR,PRINTORNOT):
    # A actual
    # P prediction
    f = np.where(np.isfinite(A*P)==1)[0]
    A = A[f]
    P = P[f]
    
    from scipy.stats import pearsonr
    MAE = np.nanmean(np.abs(A-P))
    MEDAE = np.nanmedian(np.abs(A-P))
    MAXE = np.nanmax(np.abs(A-P))
    MSE = np.nanmean((A-P)**2)
    RMSE = np.sqrt(np.nanmean((A-P)**2))
    MAPE = np.nanmean( np.abs(A-P)/np.abs(A))*100
    RSQU = r2_score(A, P)
    EXVA = 1 - np.var(A - P)/np.var(A )
    PEAR, PEAV = pearsonr(A, P)
    res = stats.linregress(A,P) 
    c = np.corrcoef(A,P)
    R2 = res.rvalue**2
    PV = res.pvalue
    CF = c[0,1]
    SL = res.slope
    SE = res.stderr
    
    ROUND_R2 = round(R2,NR)
    ROUND_PV = round(PV, NR)
    ROUND_CF = round(CF, NR)
    ROUND_SL = round(SL, NR)
    ROUND_SE = round(SE, NR)
    
    ROUND_MAE = round(MAE,NR)
    ROUND_MEDAE = round(MEDAE,NR)
    ROUND_MAXE = round(MAXE,NR)
    ROUND_MSE = round(MSE,NR)
    ROUND_RMSE = round(RMSE,NR)
    ROUND_MAPE = round(MAPE,NR)
    ROUND_RSQU = round(RSQU,NR)
    ROUND_EXVA = round(EXVA,NR)
    ROUND_PEAR = round(PEAR,NR)
    ROUND_PEAV = round(PEAV,NR)
    if PRINTORNOT == True:
        print('R²: ', ROUND_R2)
        print('p-value: ', ROUND_PV)
        print('corrcoef: ', ROUND_CF)
        print('slope: ', ROUND_SL)
        print('slope error: ', ROUND_SE)
        print('MAE: ', ROUND_MAE)
        print('MEDAE: ', ROUND_MEDAE)
        print('MAXE: ', ROUND_MAXE)
        print('MSE: ', ROUND_MSE)
        print('RMSE: ', ROUND_RMSE)
        print('MAPE: ', ROUND_MAPE,'%')
        print('RSQU: ', ROUND_RSQU)
        print('EXVA: ', ROUND_EXVA)
        print('PEAR: ', ROUND_PEAR)
        print('PEAR PV: ', ROUND_PEAV)

    if ROUNDORNOT == False:
        return MAE, MEDAE, MAXE,  MSE, RMSE,  MAPE,  RSQU,  EXVA,  PEAR,  PEAV,  R2,  PV,  CF,  SL,  SE, 
    if ROUNDORNOT == True:
        return ROUND_MAE, ROUND_MEDAE, ROUND_MAXE,  ROUND_MSE, ROUND_RMSE,  ROUND_MAPE,  ROUND_RSQU,  ROUND_EXVA,  ROUND_PEAR,  ROUND_PEAV,  ROUND_R2,  ROUND_PV,  ROUND_CF,  ROUND_SL,  ROUND_SE 


def beaning_weekly_NHOURS(y_in, marker_range, marker1, marker2,NHOURS): 
    l = len(marker_range)
    x_out = np.empty(l*52) * np.NaN
    x_out2 = np.empty(l*52) * np.NaN
    x_out3 = np.empty(l*52) * np.NaN
    x_out4 = np.empty(l*52) * np.NaN
    y_out = np.empty(l*52) * np.NaN 
    y_out2 = np.empty(l*52) * np.NaN 
    k = -1
    for i in range(0,l): 
        for w in range(1,53):
            k = k+1
            v = marker_range[i]
            x_out2[k] = v 
            x_out3[k] = w
            x_out4[k] = k
            x_out[k] = v + w/53
            fiw = np.where((marker1 == v) & (marker2 == w))[0]
            if len(fiw)>0:
                fiw = fiw[-NHOURS:]
                y_out[k] = np.nanmean(y_in[fiw])
                y_out2[k] = np.nansum(y_in[fiw])
    return x_out, x_out2, x_out3, x_out4, y_out, y_out2


def beaning_weekly(y_in, marker_range, marker1, marker2): 
    l = len(marker_range)
    x_out = np.empty(l*52) * np.NaN
    x_out2 = np.empty(l*52) * np.NaN
    x_out3 = np.empty(l*52) * np.NaN
    x_out4 = np.empty(l*52) * np.NaN
    y_out = np.empty(l*52) * np.NaN 
    y_out2 = np.empty(l*52) * np.NaN 
    k = -1
    for i in range(0,l): 
        for w in range(1,53):
            k = k+1
            v = marker_range[i]
            x_out2[k] = v 
            x_out3[k] = w
            x_out4[k] = k
            x_out[k] = v + w/53
            fiw = np.where((marker1 == v) & (marker2 == w))[0]
            if len(fiw)>0:
                y_out[k] = np.nanmean(y_in[fiw])
                y_out2[k] = np.nansum(y_in[fiw])
    return x_out, x_out2, x_out3, x_out4, y_out, y_out2


def beaning_weekly2(y_in, marker_range, marker1, marker2): 
    l = len(marker_range)
    x_out = np.empty(l*52) * np.NaN
    x_out2 = np.empty(l*52) * np.NaN
    x_out3 = np.empty(l*52) * np.NaN
    x_out4 = np.empty(l*52) * np.NaN
    y_out = np.empty(l*52) * np.NaN 
    y_out2 = np.empty(l*52) * np.NaN 
    y_out3 = np.empty(l*52) * np.NaN 
    k = -1
    for i in range(0,l): 
        for w in range(1,53):
            k = k+1
            v = marker_range[i]
            x_out2[k] = v 
            x_out3[k] = w
            x_out4[k] = k
            x_out[k] = v + w/53
            fiw = np.where((marker1 == v) & (marker2 == w))[0]
            if len(fiw)>0:
                y_out[k] = np.nanmean(y_in[fiw])
                y_out2[k] = np.nansum(y_in[fiw])
                y_out3[k] = np.nanmean(y_in[fiw]**(1/3))
    return x_out, x_out2, x_out3, x_out4, y_out, y_out2, y_out3



import matplotlib.gridspec as gridspec


def get_means_and_trends(X,Y,ROUNDORNOT,NR,PRINTORNOT):
    # X step
    # Y values
    
    f = np.where(np.isfinite(X*Y)==1)[0]
    X = X[f]
    Y = Y[f]
    if len(f)>2:
        
        MEA = np.nanmean(Y)
        MED = np.nanmedian(Y)
        MIN = np.nanmin(Y)
        MAX = np.nanmax(Y)
        STD = np.nanstd(Y)

        res = stats.linregress(X,Y)
        R2 = res.rvalue**2
        PV = res.pvalue
        SL = res.slope
        SE = res.stderr
        IT = res.intercept

        ROUND_R2 = round(R2,NR)
        ROUND_PV = round(PV, NR)
        ROUND_SL = round(SL, NR)
        ROUND_SE = round(SE, NR)
        ROUND_IT = round(IT, NR)

        ROUND_MEA = round(MEA,NR)
        ROUND_MED = round(MED,NR)
        ROUND_MAX = round(MAX,NR)
        ROUND_MIN = round(MIN,NR)
        ROUND_STD = round(STD,NR)


        if PRINTORNOT == True:
            print('R²: ', ROUND_R2)
            print('p-value: ', ROUND_PV)
            print('slope: ', ROUND_SL)
            print('slope error: ', ROUND_SE)
            print('intercept: ', ROUND_IT)

            print('MEA: ', ROUND_MAE)
            print('MED: ', ROUND_MED)
            print('MAX: ', ROUND_MAX)
            print('MIN: ', ROUND_MIN)
            print('STD: ', ROUND_STD)


        if ROUNDORNOT == False:
            return MEA, MED, MAX,  MIN, STD,  R2,  PV,  SL,  SE, IT
        if ROUNDORNOT == True:
            return ROUND_MEA, ROUND_MED, ROUND_MAX,  ROUND_MIN, ROUND_STD,  ROUND_R2,  ROUND_PV,  ROUND_SL,  ROUND_SE, ROUND_IT
    
    else: print('not enought points'); return np.nan, np.nan, np.nan,  np.nan, np.nan,  np.nan,  np.nan,  np.nan,  np.nan, np.nan

In [ ]:
def simple_linregress(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) != len(y):
        raise ValueError(f'x and y must have same length, got {len(x)} and {len(y)}')

    f = np.isfinite(x) & np.isfinite(y)

    x = x[f]
    y = y[f]

    if len(x) < 3:
        return np.nan, np.nan, np.nan, np.nan

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    rsqu = r_value**2
    ccoef = r_value

    return slope, rsqu, p_value, ccoef


def clean_xy(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) != len(y):
        raise ValueError(f'x and y must have same length, got {len(x)} and {len(y)}')

    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def get_stats(x, y):
    xx, yy = clean_xy(x, y)
    slpe, rsqu, pval, ccoef = simple_linregress(xx, yy)
    return ccoef, rsqu, pval


def get_lagged_stats(y_target, y_driver, max_lag=12):
    lags, corrs = lag_corr(y_target, y_driver, max_lag=max_lag)
    i_best = np.nanargmax(corrs)
    best_lag = lags[i_best]
    best_corr = corrs[i_best]

    if best_lag == 0:
        yt = y_target.copy()
        yd = y_driver.copy()
    else:
        yt = y_target[best_lag:]
        yd = y_driver[:-best_lag]

    ccoef, rsqu, pval = get_stats(yd, yt)

    return lags, corrs, best_lag, best_corr, rsqu, pval



# ----------------------------
# FUNCTION
# ----------------------------
def compute_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        return np.nan, np.nan, np.nan

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 5:
        return np.nan, np.nan, np.nan

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    return r_value, r_value**2, p_value


def select_metric(cc, r2, pval):
    if use_pfilter and (not np.isfinite(pval) or pval > p_thresh):
        return np.nan

    if metric == 'R2':
        return r2
    elif metric == 'cc':
        return cc
    elif metric == 'pval':
        return pval
    else:
        return np.nan